In [ ]:
import grp
from langchain.tools import tool
from typing import Annotated

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_ollama import ChatOllama

from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator

from langchain.messages import SystemMessage
from langchain.messages import ToolMessage

from typing import Literal
from langgraph.graph import StateGraph, START, END

#gemma4:31b-cloud

In [ ]:
model_name = "gemma4:31b-cloud"

lib_phrases = ""
ntss_phrases = ""
rental_phrases = ""

lib_truth = dict()
rent_truth = dict()
ntss_truth = dict()

In [ ]:
#Helpers
def score_response(response: str, ground_truth:dict[str,str]) -> tuple[float, float, float]:
    """Scores the model response against ground truth and returns precision, recall, and f1"""
    predictions = {} # Logic to parse response goes here
    tp = sum(1 for k, v in predictions.items() if v == ground_truth.get(k))
    fp = sum(1 for k, v in predictions.items() if v != ground_truth.get(k))
    fn = sum(1 for k in ground_truth if k not in predictions)
    c_matrix = {"TP": tp, "FP": fp, "FN": fn}
    precision = calc_precision(c_matrix) if (tp + fp) > 0 else 0.0
    recall = calc_recall(c_matrix) if (tp + fn) > 0 else 0.0
    f1 = calculate_f1(precision, recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def build_eval_prompt(state:dict[str,str]) -> str:
    """Constructs a prompt for the optimizer to improve the current prompt based on eval results and ground truths"""
    # Including ground truths allows the model to perform error analysis
    ground_truths = {
        'library': lib_truth,
        'rental': rent_truth,
        'ntss': ntss_truth
    }
    return f"Current Prompt: {state['current_prompt']}\n\nEvaluation Results: {state['lib_eval']} {state['rental_eval']} {state['ntss_eval']}\n\nGround Truths: {ground_truths}\n\nBased on the results and the ground truths, please analyze where the predictions went wrong and provide an improved version of the prompt to increase the F1 score."

def generate_confusion_matrix(pred, targets):
    """Takes model predictions and ground truths and returns a confusion matrix as a dict"""
    return { "TP":0, "TN":0, "FP":0, "FN":0 }

def calc_precision(c_matrix):
    """Given confusion matrix, calculate how precise the models predictions are"""
    return c_matrix["TP"] / (c_matrix["TP"] + c_matrix["FP"])

def calc_recall(c_matrix):
    """Given confusion matrix, calculate the models ability to identify all positive cases in a dataset"""
    return c_matrix["TP"] / (c_matrix["TP"] + c_matrix["FN"])

def calculate_f1(precision, recall):
    """Calculates F1 score for classification task"""
    return 2 * ((precision * recall) / (precision + recall))

In [ ]:
model = ChatOllama(
    model=model_name,
    temperature=0
)



In [ ]:
class OptimizerState(TypedDict):
    current_prompt: str
    current_trial: int
    optimized_prompt: str
    rental_eval: dict[str,str]
    ntss_eval: dict[str,str]
    lib_eval: dict[str,str]

In [ ]:
## Nodes
def run_lib_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", lib_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "lib_eval":{
            "response":model_response
        }
    }

def run_rental_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", rental_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "rental_eval":{
            "response":model_response
        }
    }

def run_ntss_classifier(state: dict):
    """Builds full lib prompt and asks model to classify domain phrases"""

    prompt_template = state["current_prompt"]
    full_prompt = str(prompt_template).replace("<DOMAIN PHRASES>", ntss_phrases)
    model_response  = model.invoke(full_prompt)

    return {
        "ntss_eval":{
            "response":model_response
        }
    }

def score(state: dict):
    """Score evals for each dataset adding precision, recall, and f1 to state for each"""
    lib_data = state["lib_eval"]
    lib_p, lib_r, lib_f1 = score_response(lib_data["response"], lib_truth)

    rent_data = state["rental_eval"]
    rent_p, rent_r, rent_f1 = score_response(rent_data["response"], rent_truth)

    ntss_data = state["ntss_eval"]
    ntss_p, ntss_r, ntss_f1 = score_response(ntss_data["response"], ntss_truth)

    return {
        "lib_eval":{
            "precision":lib_p,
            "recall":lib_r,
            "f1":lib_f1
        },
        "rental_eval":{
            "precision":rent_p,
            "recall":rent_r,
            "f1":rent_f1
        },
        "ntss_eval":{
            "precision":ntss_p,
            "recall":ntss_r,
            "f1":ntss_f1
        }
    }

def optimize(state: dict):
    """Read eval data, prompt, and ground truths and optimize prompt"""
    eval_prompt = build_eval_prompt(state)
    improved_prompt = model.invoke(eval_prompt)

    return {
        "optimized_prompt":improved_prompt
    }

def checkpoint(state:dict):
    """Write optimized prompt to classification folder"""
    # located in: /prompts/automated_propmt_evolution/classification/t{num_trial}.md
    # Need to understand what the current trial is
    with open(f"t{state["current_trial"]}", 'w', encoding='utf-8') as f:
        f.write(state["optimized_prompt"])

In [ ]:
# def should_continue(state:MessageState) -> Literal["tool_node", END]:
#     """Decide if we should continue the loop or stop based upon the L"""
#     messages = state["messages"]
#     last_message = messages[-1]
#
#     if last_message.tool_calls:
#         return "tool_node"
#
#     return END


In [ ]:
# agent_builder = StateGraph(MessageState)
#
# agent_builder.add_node("llm_call", llm_call)
# agent_builder.add_node("tool_node", tool_node)
#
# agent_builder.add_edge(START, "llm_call")
# agent_builder.add_conditional_edges(
#     "llm_call",
#     should_continue,
#     ["tool_node", END]
# )
#
# agent_builder.add_edge("tool_node","llm_call")
#
# agent=agent_builder.compile()
#
# from IPython.display import Image, display
# display(Image(agent.get_graph(xray=True).draw_mermaid_png()))
#
# #Invoke
# from langchain.messages import HumanMessage
# messages = [HumanMessage(content="Do something")]
# messages = agent.invoke({"messages": messages})
# for m in messages["messages"]:
#     m.pretty_print()